In [ ]:
# FILE: 01_Data_Preparation.ipynb
# NEW CELL 1

# --- Install all required packages for the project ---
%pip install pandas geopandas rasterio shapely fiona pyproj rapidfuzz xarray netCDF4 dask matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.5 MB/s eta 0:00:00


In [ ]:
# FILE: 01_Data_Preparation.ipynb
# CELL 2 (REPLACEMENT - for setup)

from google.colab import drive

# --- Mount Drive FIRST ---
drive.mount('/content/drive', force_remount=True)

# --- Change into the project directory SECOND ---
%cd "/content/drive/MyDrive/IndiaProject_Colab"

# --- Run the setup notebook THIRD ---
# This will now work because we are in the correct directory.
%run 00_Functions_and_Setup.ipynb

Mounted at /content/drive
/content/drive/MyDrive/IndiaProject_Colab


In [ ]:
# FILE: 01_Data_Preparation.ipynb
# CELL 2 (CORRECTED AND RESTORED LOGIC)

# --- Step 1: Match District Geographies (with full fuzzy logic) ---
print("--- Starting Step 1: Matching District Geographies ---")

# --- 1. Load Data ---
crop = pd.read_csv(CSV_CROP)
dist = gpd.read_file(SHP_DIST).to_crs(4326)

# --- 2. Prepare Crop Data's 'code_map' ---
state_csv_col = find_first(crop.columns, ["State Name","STATE"])
dist_csv_col  = find_first(crop.columns, ["Dist Name","DISTRICT"])
code_csv_col  = find_first(crop.columns, ["Dist Code"])
code_map = (crop[[state_csv_col, dist_csv_col, code_csv_col]].drop_duplicates()
    .assign(state_norm=lambda d: d[state_csv_col].map(canon_state), dist_norm =lambda d: d[dist_csv_col].map(_norm))
    [["state_norm","dist_norm", "Dist Code"]].dropna(subset=["state_norm","dist_norm"])
    .drop_duplicates(["state_norm","dist_norm"]))
choices_by_state = (code_map.groupby("state_norm").agg(choices=("dist_norm", list), codes=("Dist Code", list)))

# --- 3. Prepare Shapefile Data ---
state_shp_col = find_first(dist.columns, ["STATE","ST_NM","NAME_1"])
dist_shp_col  = find_first(dist.columns, ["DISTRICT","DIST_NAME","NAME_2"])
dist = dist.rename(columns={state_shp_col: "state_shp", dist_shp_col: "dist_shp"})
dist["state_norm"] = dist["state_shp"].map(canon_state)
dist["dist_norm"]  = dist["dist_shp"].map(_norm)

# --- 4. Perform Exact Match ---
dist_join = dist.merge(code_map, on=["state_norm","dist_norm"], how="left")
print(f"Exact matches: {dist_join['Dist Code'].notna().sum()} / {len(dist_join)}")

# --- 5. Restore the FULL Fuzzy Match Logic for Unmatched Polygons ---
unmatched_idx = dist_join.index[dist_join["Dist Code"].isna()].tolist()
if unmatched_idx:
    print(f"Attempting full fuzzy match for the remaining {len(unmatched_idx)} polygons...")

    # <<< RESTORED: Find all possible variant name columns
    variant_cols = sorted(list(set([c for c in dist_join.columns if isinstance(dist_join[c].dtype, object) and c != 'geometry'])))

    fuzzy_results = []
    for i in unmatched_idx:
        row = dist_join.loc[i]
        st = row["state_norm"]
        if st not in choices_by_state.index: continue

        cand_names, cand_codes = choices_by_state.loc[st, "choices"], choices_by_state.loc[st, "codes"]

        # <<< RESTORED: Gather all variant names from all text columns
        variants = [_norm(row["dist_shp"])]
        for c in variant_cols:
            variants += _split_variants(row.get(c))
        variants = sorted(list(set(variants))) # Get unique variants

        best = (-1, None, None, "none")
        # <<< RESTORED: Two-pass fuzzy scoring
        for v in variants:
            # First pass with WRatio
            res = process.extractOne(v, cand_names, scorer=fuzz.WRatio)
            if res and res[1] > best[0]:
                idx = cand_names.index(res[0])
                best = (int(res[1]), cand_codes[idx], res[0], "wratio")
        if best[0] < 90: # If WRatio is not great, try token_set_ratio
            for v in variants:
                res = process.extractOne(v, cand_names, scorer=fuzz.token_set_ratio)
                if res and res[1] > best[0]:
                    idx = cand_names.index(res[0])
                    best = (int(res[1]), cand_codes[idx], res[0], "token_set")

        score, code, name, method = best
        if score >= 90:
             fuzzy_results.append({'index': i, 'Dist Code': code})

    if fuzzy_results:
        fix = pd.DataFrame(fuzzy_results).set_index('index')
        dist_join.update(fix)
        print(f"Accepted fuzzy matches: {len(fix)}")

# --- 6. Finalize and Save Output ---
districts_joined = dist_join.dropna(subset=["Dist Code"]).copy()
districts_joined["Dist Code"] = districts_joined["Dist Code"].astype(int)
districts_joined.to_file(DIST_GPKG, driver="GPKG")
print(f"--- Step 1 Complete: Saved '{DIST_GPKG}' with {len(districts_joined)} matched districts. ---")

--- Starting Step 1: Matching District Geographies ---
Exact matches: 229 / 310
Attempting full fuzzy match for the remaining 81 polygons...
Accepted fuzzy matches: 81
--- Step 1 Complete: Saved '/content/drive/MyDrive/IndiaProject_Colab/data/processed/districts_joined.gpkg' with 310 matched districts. ---


In [ ]:
# FILE: 01_Data_Preparation.ipynb
# REPLACEMENT for Step 2 Cell

print("\n--- Starting Step 2: Preparing Master Crop Data (Yield + Area + Irrigation) ---")

# --- 1. Define Column Mappings ---
# Yield Columns (from Main CSV)
yield_cols = {
    'RICE YIELD (Kg per ha)': 'rice',
    'MAIZE YIELD (Kg per ha)': 'maize',
    'KHARIF SORGHUM YIELD (Kg per ha)': 'sorghum_kharif',
    'RABI SORGHUM YIELD (Kg per ha)': 'sorghum_rabi'
}

# Total Area Columns (from Main CSV)
area_cols = {
    'RICE AREA (1000 ha)': 'rice',
    'MAIZE AREA (1000 ha)': 'maize',
    'KHARIF SORGHUM AREA (1000 ha)': 'sorghum_kharif',
    'RABI SORGHUM AREA (1000 ha)': 'sorghum_rabi'
}

# --- 2. Helper Function to Process (Melt) Variables ---
def process_variable(df, col_map, value_name, variable_type):
    """Melts a specific set of columns (yield, area, or irrig) into long format."""

    # Dynamic Column Finder (Handles variations in headers)
    # We construct the map based on what is actually in the DF
    actual_map = {}

    # 1. Try exact matches first
    for csv_col, target_name in col_map.items():
        if csv_col in df.columns:
            actual_map[csv_col] = target_name

    # 2. If Maize irrigation is missing, try fuzzy search
    if variable_type == 'irrigation' and 'maize' not in actual_map.values():
        print("  > Searching for Maize Irrigation column...")
        # Look for columns containing both "MAIZE" and "IRRI"
        candidates = [c for c in df.columns if "MAIZE" in c.upper() and "IRRI" in c.upper()]
        if candidates:
            print(f"  > Found match: '{candidates[0]}' -> 'maize'")
            actual_map[candidates[0]] = 'maize' # <--- CRITICAL FIX: Maps to 'maize', not 'MAIZE IRRI'
        else:
            print("  > [!] WARNING: Could not find Maize Irrigation column.")

    if not actual_map:
        print(f"  [!] WARNING: No columns found for {variable_type}!")
        return pd.DataFrame()

    # 3. Subset and Rename
    subset = df[["Dist Code", "Year", "State Name", "Dist Name"] + list(actual_map.keys())].copy()
    subset.rename(columns=actual_map, inplace=True)

    # 4. Melt
    melted = subset.melt(
        id_vars=["Dist Code", "Year", "State Name", "Dist Name"],
        value_vars=list(actual_map.values()),
        var_name="crop",
        value_name=value_name
    )
    return melted

# --- 3. Process Main Crop CSV (Yield + Total Area) ---
print(f"Loading Main Crop CSV: {CSV_CROP}...")
df_main = pd.read_csv(CSV_CROP, low_memory=False)
df_main = df_main[pd.to_numeric(df_main["Year"], errors="coerce").between(2000, 2017)].copy()

print("  Processing Yields...")
df_yield = process_variable(df_main, yield_cols, "yield_kg_per_ha", "yield")

print("  Processing Total Areas...")
df_area = process_variable(df_main, area_cols, "total_area_1000ha", "area")

# Merge Yield and Area
print("  Merging Yield and Area data...")
df_combined = pd.merge(
    df_yield,
    df_area,
    on=["Dist Code", "Year", "State Name", "Dist Name", "crop"],
    how="outer"
)

# --- 4. Process Irrigation CSV ---
print(f"Loading Irrigation CSV: {CSV_IRRIG}...")

# Base mapping for Irrigation (Maize will be handled dynamically if exact name fails)
irrig_cols_base = {
    'RICE IRRIGATED AREA (1000 ha)': 'rice',
    'MAIZE IRRIGATED AREA (1000 ha)': 'maize', # <--- Standard Name
    'KHARIF SORGHUM IRRIGATED AREA (1000 ha)': 'sorghum_kharif',
    'RABI SORGHUM IRRIGATED AREA (1000 ha)': 'sorghum_rabi'
}

try:
    df_irrig_raw = pd.read_csv(CSV_IRRIG, low_memory=False)
    df_irrig_raw = df_irrig_raw[pd.to_numeric(df_irrig_raw["Year"], errors="coerce").between(2000, 2017)].copy()

    # CLEANING: Replace -1 with NaN
    df_irrig_raw.replace(-1, np.nan, inplace=True)

    print("  Processing Irrigated Areas...")
    df_irrig = process_variable(df_irrig_raw, irrig_cols_base, "irrig_area_1000ha", "irrigation")

    # Merge Irrigation into Combined Data
    print(f"  Merging Irrigation data... (Records: {len(df_irrig)})")

    # Check crops before merge
    print(f"  Crops found in Irrigation Data: {df_irrig['crop'].unique()}")

    final_df = pd.merge(
        df_combined,
        df_irrig[["Dist Code", "Year", "crop", "irrig_area_1000ha"]],
        on=["Dist Code", "Year", "crop"],
        how="left"
    )

except FileNotFoundError:
    print(f"[!] ERROR: Irrigation file not found at {CSV_IRRIG}")
    final_df = df_combined
    final_df["irrig_area_1000ha"] = np.nan

# --- 5. Final Cleanup and Save ---
final_df.rename(columns={"Year": "year", "State Name": "state", "Dist Name": "district"}, inplace=True)
numeric_cols = ["yield_kg_per_ha", "total_area_1000ha", "irrig_area_1000ha", "Dist Code", "year"]
for c in numeric_cols:
    final_df[c] = pd.to_numeric(final_df[c], errors="coerce")

final_df.sort_values(["Dist Code", "year", "crop"], inplace=True)
final_df.to_csv(YLD_FILE, index=False)
print(f"\n--- Step 2 Complete: Saved Master File to '{YLD_FILE}' ---")


--- Starting Step 2: Preparing Master Crop Data (Yield + Area + Irrigation) ---
Loading Main Crop CSV: /content/drive/MyDrive/IndiaProject_Colab/data/raw/ICRISAT-District Level Data_Apportioned.csv...
  Processing Yields...
  Processing Total Areas...
  Merging Yield and Area data...
Loading Irrigation CSV: /content/drive/MyDrive/IndiaProject_Colab/data/raw/ICRISAT-District Level Irrigation Data.csv...
  Processing Irrigated Areas...
  Merging Irrigation data... (Records: 21312)
  Crops found in Irrigation Data: ['rice' 'maize' 'sorghum_kharif' 'sorghum_rabi']

--- Step 2 Complete: Saved Master File to '/content/drive/MyDrive/IndiaProject_Colab/data/processed/icrisat_yield_area_irrig_4crops_long_2000_2017.csv' ---
